# CSP Pipeline

This notebook loads the Layer E candidate slots, models the scheduling problem as a **Constraint Satisfaction Problem** (one variable per satellite), reduces it exactly and decomposes it into sub-problems small enough for the QPU. It is the CSP counterpart of the MWIS pipeline (`02.A_mwis_pipeline.ipynb`): the stages, budgets and artifacts mirror it, so the two producers can be compared downstream.

| Stage | Section | Status |
|---|---|---|
| Load candidate slots | 2 | done |
| Build the CSP model | 3 | done |
| Exact reduction: dominance, connected components | 4 | done |
| Exact decomposition into QPU-sized terminals (separators of satellites) | 5 | done |
| Heuristic fallback of incomplete leaves (greedy + upper bound) | 6 | done, per leaf |
| QUBO translation of the terminals (QUBO dict + index map) | 7 | done |
| Classical evaluation: baseline schedule, component status, upper bounds | 8 | done |
| Export: CSP structure, sub-problems, QUBOs, baseline, reconstruction metadata | 9 | done |
| Annealing of the terminal QUBOs and reconstruction, MILP fallback | annealing notebook | to do (CSP runs not supported yet) |

## 1. Configuration

All run parameters (input file, run name, QPU limits, decomposition budgets) are read from `results/_config.json` (or from the file in the `SWE2_CONFIG` environment variable). Set `run_name_csp` to select the CSP experiment and `csp_slot_csv` to select its input; the CSP graphics notebook uses the same settings. The decomposition budgets use the same keys as the MWIS pipeline, so both run under the same limits; `csp_qubo_encoding` (optional, default `domain_wall`) selects the QUBO encoding the terminals are sized with. A copy of the configuration used is saved as `run_config_log.json` in `results/runs/csp/pipeline_results/<run_name_csp>`.

In [ ]:
from __future__ import annotations

import gzip
import json
import os
import sys
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import pandas as pd
pd.set_option("display.max_columns", None)


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"

# project modules live in `src/`
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from CSP import CSP, CSP_Variable, Timeslot
from CSP_decomposition import dominance_reduction, split_connected_components, decompose_csp
from CSP_to_QUBO import csp_to_qubo


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS and QUBO notebooks)
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: candidate slots produced by the data-wrangling notebooks
CSP_SLOT_CSV = PROJECT_ROOT / run_config["csp_slot_csv"]

# CSP experiment name and output directory
RUN_NAME = run_config["run_name_csp"]
RUN_DIR = RESULTS_DIR / "runs" / "csp" / "pipeline_results" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# QPU physical constraints: every terminal sub-problem must fit them
MAX_QPU_TERMINAL_NODES = run_config["max_qpu_terminal_nodes"]
MAX_QPU_TERMINAL_COUPLERS = run_config["max_qpu_terminal_couplers"]

# exact decomposition budgets (same keys and values as the MWIS pipeline)
MAX_SEPARATOR_SIZE = run_config["max_separator_size"]
MAX_SEPARATOR_CANDIDATE_POOL = run_config["max_separator_candidate_pool"]
MAX_SEPARATOR_STATES = run_config["max_separator_states"]
MAX_RECURSION_DEPTH = run_config["max_recursion_depth"]
MAX_DECOMPOSITION_NODES = run_config["max_decomposition_nodes"]

# QUBO encoding of the satellite variables, used to size the terminals
QUBO_ENCODING = run_config.get("csp_qubo_encoding", "domain_wall")

# logging environment configuration
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"CSP experiment: {RUN_NAME}")
print(f"Input dataset: {CSP_SLOT_CSV.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")
print(f"QPU limits: {MAX_QPU_TERMINAL_NODES} qubits, {MAX_QPU_TERMINAL_COUPLERS} couplers ({QUBO_ENCODING} encoding)")

PHASE_PROFILE_PATH = RUN_DIR / f'csp_phase_profile_{time.strftime("%Y%m%d_%H%M%S")}.jsonl'
PIPELINE_STARTED = time.perf_counter()

def record_phase_timing(phase: str, started: float, **details: Any) -> float:
    """
    Helper function.
    Get how much time the specified phase takes to run, and store it in the log file.
    """
    elapsed = time.perf_counter() - started
    event = {"phase": phase, "elapsed_seconds": round(elapsed, 3), **details}
    with PHASE_PROFILE_PATH.open("a", encoding="utf-8") as profile_file:
        profile_file.write(json.dumps(event) + "\n")
        profile_file.flush()
    print(f"[CSP timing] {phase}: {elapsed:.2f}s", flush=True)
    return elapsed

## 2. Load candidate slots

The data-wrangling notebooks already created weighted candidate contacts: each raw visibility window was discretised into fixed-length candidate slots, one every `discretization_step_seconds`. Every slot inherits the priority, urgency and weight of its own parent window.

In [ ]:
REQUIRED_COLUMNS = [
    "slot_id", "parent_window_id", "satellite_id", "satellite_name", "antenna_id", "station_id",
    "start_time_hours", "end_time_hours", "duration_hours",
    "reconfiguration_time_hours", "priority", "urgency", "weight",
]

raw_slots = pd.read_csv(CSP_SLOT_CSV)
slots = raw_slots[REQUIRED_COLUMNS].copy()

print({
    "candidate_slots": len(slots),
    "satellites": slots["satellite_name"].nunique(),
    "antennas": slots["antenna_id"].nunique(),
    "source_intervals": slots["parent_window_id"].nunique(),
    "time_horizon_hours": (round(slots["start_time_hours"].min(), 3), round(slots["end_time_hours"].max(), 3)),
})

# slot weights are not stored in the CSP: reductions and QUBOs look them up by slot_id
SLOT_WEIGHT = slots.set_index("slot_id")["weight"].astype(float).to_dict()

## 3. Translating to CSP

| Entity | CSP element |
|---|---|
| Satellite *S* | `CSP_Variable` (one per unique satellite) |
| Candidate slot at antenna *A* | `Timeslot` holding its `slot_id`, antenna, start/end and reconfiguration time (integer seconds) |
| All candidate slots of satellite *S* | Domain of variable *S* |
| Two satellites have a pair of slots on the same antenna that overlap, or leave less than the reconfiguration time between them (`Timeslot.overlap`) | *Constraint*: they cannot both take those slots |

The conflicts between slots are computed once, when the variables are added, and stored in the CSP's `ConflictIndex`; constraints are read from it and from the current domains, so the reductions below only remove values and every sub-problem shares the index. Slot weights are not stored in the CSP: they are looked up by `slot_id`.

Antenna conflicts follow the same rule as the MWIS formulation. The model assumption that differs from MWIS is:
- each satellite receives **at most one** contact over the whole horizon (it may also stay unscheduled).

In [ ]:
phase_started = time.perf_counter()

# build the CSP: one variable per satellite, its domain is the slots it may use
# (values are loaded before add_var, which registers them in the conflict index)
csp = CSP()
for sat_name, sat_rows in slots.groupby("satellite_name", sort=True):
    var = CSP_Variable(name=sat_name)
    for row in sat_rows.itertuples(index=False):
        var.add_value(Timeslot.from_hours(
            row.slot_id, row.antenna_id, row.reconfiguration_time_hours,
            row.start_time_hours, row.end_time_hours,
        ))
    csp.add_var(var)

record_phase_timing("csp_build", phase_started, variables=len(csp.vars), constraints=csp.num_constraints())


# --- prettier summary print ---
def print_summary(label, summ):
    print(f'\n{"-"*46}')
    print(f'  {label}')
    print(f'{"-"*46}')
    for k, v in summ.items():
        val = f'{v:.4f}' if isinstance(v, float) else str(int(v))
        print(f'  {k:<24} {val}')

csp_summary = csp.compute_summary()
print_summary("CSP  —  summary", csp_summary)

## 4. Exact reduction

Two exact reductions shrink the problem before the decomposition; neither can lose the optimum.

**Dominance.** A slot *a* of a satellite is removed when another slot *b* of the same satellite weighs at least as much and conflicts with a subset of the slots *a* conflicts with: any schedule using *a* stays feasible, and is not worse, using *b*. The rule is repeated until nothing changes, since every removal shrinks the conflicts of the other satellites. Unlike AC-3, it is exact even when satellites may stay unscheduled.

**Connected components.** Satellites with no conflicting slots are independent: the constraint graph splits into components, each a sub-problem solved on its own and merged back at the end (no constraint is cut).

The table compares the QUBO size of the whole problem before and after dominance, with both encodings: the domain-wall encoding turns the *at most one slot per satellite* cliques into chains.

In [ ]:
phase_started = time.perf_counter()

def qubo_size(problem: CSP, encoding: str) -> tuple[int, int]:
    """(qubits, couplers) of the QUBO of a CSP with the given encoding."""
    Q, index_map = csp_to_qubo(problem, SLOT_WEIGHT, encoding=encoding)
    return len(index_map), sum(1 for i, j in Q if i != j)

# dominance works on a copy: the original CSP is kept for the structure export
reduced_csp = csp.copy()
dominated_slots = dominance_reduction(reduced_csp, SLOT_WEIGHT)

# independent sub-problems, largest first
components = split_connected_components(reduced_csp)
record_phase_timing("exact_reduction", phase_started, dominated_slots=dominated_slots, components=len(components))

size_rows = []
for stage, problem in (("before dominance", csp), ("after dominance", reduced_csp)):
    for encoding in ("one_hot", "domain_wall"):
        qubits, couplers = qubo_size(problem, encoding)
        size_rows.append({"stage": stage, "encoding": encoding, "qubits": qubits, "couplers": couplers,
                          "constraints": problem.num_constraints()})

component_slots = [sum(len(v.domain) for v in c.vars) for c in components]
print(f"Dominated slots removed : {dominated_slots}")
print(f"Connected components    : {len(components)}")
print(f"  Largest component     : {len(components[0].vars)} satellites, {component_slots[0]} slots")
print(f"  Single-satellite ones : {sum(1 for c in components if len(c.vars) == 1)}")
pd.DataFrame(size_rows)

## 5. Exact decomposition

Every component larger than the QPU limits is decomposed with `decompose_csp`, the CSP counterpart of the MWIS separator decomposition. Each node of the decomposition tree applies the exact reductions again and becomes:

| Node type | Meaning |
|---|---|
| `terminal` | the QUBO of the sub-problem (with `QUBO_ENCODING`) fits the QPU limits, or nothing is left after the reductions |
| `independent_sum` | the sub-problem splits into independent pieces, solved separately and summed |
| `separator` | one branch per feasible state of a few *separator satellites* (each takes one of its slots, or none); every branch forward-checks the choice (removes the slots conflicting with it) and decomposes what is left |
| `incomplete` | a budget was exceeded: the leaf needs a heuristic fallback (section 6) |

The separator is the affordable cut (its states fit the remaining budget) that leaves the smallest sub-problems: candidates are articulation points, pairs of satellites and small cuts among the highest-degree satellites. Considering every state preserves the optimum: a component whose tree has no incomplete leaf is decomposed exactly. Budgets are counted per component, as in the MWIS pipeline.

On dense instances this step is the slowest of the notebook (each branch repeats the reductions on its own sub-problem).

In [ ]:
phase_started = time.perf_counter()
decompositions = []
for index, component in enumerate(components, start=1):
    component_id = f"component_{index:03d}"
    component_started = time.perf_counter()
    decomposition = decompose_csp(
        component, SLOT_WEIGHT, component_id,
        max_qubits=MAX_QPU_TERMINAL_NODES,
        max_couplers=MAX_QPU_TERMINAL_COUPLERS,
        encoding=QUBO_ENCODING,
        max_separator_size=MAX_SEPARATOR_SIZE,
        max_candidate_pool=MAX_SEPARATOR_CANDIDATE_POOL,
        max_separator_states=MAX_SEPARATOR_STATES,
        max_recursion_depth=MAX_RECURSION_DEPTH,
        max_decomposition_nodes=MAX_DECOMPOSITION_NODES,
    )
    decompositions.append(decomposition)
    if len(decomposition["nodes"]) > 1:
        record_phase_timing("component_decomposition", component_started, component_id=component_id,
                            nodes=len(decomposition["nodes"]), status=decomposition["status"])
record_phase_timing("exact_decomposition", phase_started, components=len(decompositions))

# one summary row per component (same role as the MWIS decomposition_summary.csv)
decomposition_rows = []
for component, decomposition in zip(components, decompositions):
    root = decomposition["root"]
    terminals = [n for n in decomposition["terminals"] if n["qubits"] > 0]
    incomplete = decomposition["incomplete"]
    decomposition_rows.append({
        "component_id": decomposition["component_id"],
        "status": decomposition["status"],
        "root_type": root["node_type"],
        "satellites": len(component.vars),
        "input_qubits": root["input_qubits"],
        "terminal_subproblems": len(terminals),
        "largest_terminal_qubits": max((n["qubits"] for n in terminals), default=0),
        "incomplete_nodes": len(incomplete),
        "largest_incomplete_qubits": max((n["qubits"] for n in incomplete), default=0),
        "separator_states": decomposition["separator_state_count"],
        "cache_hits": decomposition["cache_hits"],
        "max_depth": decomposition["max_depth"],
    })
decomposition_summary = pd.DataFrame(decomposition_rows)
decomposition_summary.to_csv(RUN_DIR / "decomposition_summary.csv", index=False)

all_nodes = [n for d in decompositions for n in d["nodes"]]
print({
    "components": len(decompositions),
    "complete_components": sum(d["status"] == "complete" for d in decompositions),
    "tree_nodes": len(all_nodes),
    "node_types": dict(Counter(n["node_type"] for n in all_nodes)),
})
decomposition_summary.head(10)

## 6. Heuristic fallback of incomplete leaves

As in the MWIS pipeline, an incomplete leaf is not hidden: `decompose_csp` gives it a `heuristic_solution` made of
- a **feasible schedule**: a deterministic greedy (slots by weight / (1 + degree), a slot taken when its satellite is still free and it conflicts with nothing taken), plus the leaf's exact forced choices;
- a **conservative upper bound** on the leaf's optimum: every satellite takes at most one slot, so the sum of each satellite's heaviest slot (tighter than the MWIS sum of all weights).

The gap between the two bounds how far the leaf's schedule can be from its optimum. Combining the leaves into one schedule and one global bound requires the tree evaluation (section 9).

In [ ]:
incomplete_rows = [
    {
        "node_id": n["decomposition_node_id"],
        "component_id": n["original_component_id"],
        "depth": n["depth"],
        "reason": n["reason"],
        "qubits": n["qubits"],
        "heuristic_objective": n["heuristic_solution"]["objective"],
        "upper_bound": n["heuristic_solution"]["upper_bound"],
        "gap": n["heuristic_solution"]["upper_bound"] - n["heuristic_solution"]["objective"],
    }
    for d in decompositions for n in d["incomplete"]
]
incomplete_leaves = pd.DataFrame(incomplete_rows)
if incomplete_leaves.empty:
    print("No incomplete leaf: every component is decomposed exactly.")
else:
    print(f"Incomplete leaves: {len(incomplete_leaves)}")
    print("Reasons:", dict(Counter(incomplete_leaves["reason"])))
    print(f"Largest incomplete leaf: {incomplete_leaves['qubits'].max()} qubits")
    print(f"Relative gap (all leaves): {incomplete_leaves['gap'].sum() / incomplete_leaves['upper_bound'].sum():.1%}")
incomplete_leaves.head(10)

## 7. QUBO translation of the terminals

Every non-empty terminal is translated into a QUBO with `csp_to_qubo` (encoding `QUBO_ENCODING`, `domain_wall` by default): the reward term favours each slot by its normalised weight, a penalty forbids conflicting slots of different satellites, and the encoding keeps at most one slot per satellite (a chain penalty for domain-wall, pairwise penalties for one-hot). Penalties are set above the largest reward, so the QUBO minimum is a feasible schedule and an optimal one.

Each QUBO is saved with its `index_map` (`{slot_id: [bit, next_bit]}`): a slot is chosen in a sample when `sample[bit] - sample[next_bit]` equals 1 (`next_bit` is null for one-hot and at the end of a domain-wall chain), which is what `decode_sample` computes. Each record also carries `slot_weights` and `conflicts` (the slot pairs that cannot be selected together: antenna conflicts and any two slots of the same satellite), so a decoded sample can be scored and checked without the CSP. Together with the terminal's forced choices (slot ids), this is everything needed to turn an annealing sample back into slots of the original problem.

In [ ]:
phase_started = time.perf_counter()

def terminal_csp(leaf: dict[str, Any]) -> CSP:
    """
    Rebuild the sub-problem of a tree leaf from the reduced CSP: a cheap subproblem sharing the
    conflict index, restricted to the leaf's remaining slots.
    """
    sub_csp = reduced_csp.subproblem(leaf["domains"])
    for var in sub_csp.vars:
        keep = set(leaf["domains"][var.name])
        var.domain = {t for t in var.domain if t.slot_id in keep}
    return sub_csp

terminal_qubos = []
for d in decompositions:
    for leaf in d["terminals"]:
        if leaf["qubits"] == 0:
            continue    # resolved by the exact preprocessing: nothing to anneal
        sub_csp = terminal_csp(leaf)
        Q, index_map = csp_to_qubo(sub_csp, SLOT_WEIGHT, encoding=QUBO_ENCODING)
        # slot pairs that cannot be selected together: antenna conflicts and any two slots of a satellite
        conflicts = {tuple(sorted((t.slot_id, u.slot_id)))
                     for var in sub_csp.vars for t in var.domain
                     for u in sub_csp.index.conflicts[t] if u.slot_id in index_map}
        for var in sub_csp.vars:
            conflicts.update(tuple(sorted(pair)) for pair in combinations([t.slot_id for t in var.domain], 2))
        terminal_qubos.append({
            "terminal_id": leaf["decomposition_node_id"],
            "original_component_id": leaf["original_component_id"],
            "encoding": QUBO_ENCODING,
            "qubits": len(index_map),
            "couplers": sum(1 for i, j in Q if i != j),
            "forced_selected": sorted(leaf["forced_selected"].values()),   # exact choices (slot ids), added to the decoded sample
            "forced_objective": leaf["forced_objective"],
            "qubo": [[i, j, coefficient] for (i, j), coefficient in sorted(Q.items())],   # {(i, j): coeff} as rows
            "index_map": {slot_id: list(bits) for slot_id, bits in index_map.items()},  # slot_id -> [bit, next_bit]
            # to score a decoded sample: slot weights and the slot pairs that cannot be selected together
            "slot_weights": {slot_id: SLOT_WEIGHT[slot_id] for slot_id in index_map},
            "conflicts": sorted(list(pair) for pair in conflicts),
        })

(RUN_DIR / "csp_terminal_qubos.json").write_text(json.dumps(terminal_qubos), encoding="utf-8")
record_phase_timing("qubo_translation", phase_started, terminals=len(terminal_qubos))

qubo_sizes = pd.DataFrame([{k: t[k] for k in ("terminal_id", "qubits", "couplers")} for t in terminal_qubos])
print(f"Terminal QUBOs: {len(terminal_qubos)} ({QUBO_ENCODING} encoding)")
if not qubo_sizes.empty:
    print(f"  qubits   max {qubo_sizes['qubits'].max()} (limit {MAX_QPU_TERMINAL_NODES})")
    print(f"  couplers max {qubo_sizes['couplers'].max()} (limit {MAX_QPU_TERMINAL_COUPLERS})")
qubo_sizes.head(10)

## 8. Classical evaluation of the decomposition

As in the MWIS pipeline, the decomposition trees are evaluated classically before annealing. This gives a **baseline schedule**, the **status** of every component (`optimal` when its value is certified, `heuristic_feasible` otherwise) and **upper bounds** on the optimum, which the annealing notebook uses to report its gap and its MILP fallback uses as a starting bound.

Each tree is evaluated from the leaves upward:

| Node | Value | Upper bound |
|---|---|---|
| `terminal` | forced choices + exact optimum of the terminal (branch and bound over its satellites; terminals fit the QPU, so this is fast) | its value (exact) |
| `incomplete` | its `heuristic_solution` (greedy + forced choices) | the conservative bound of section 6 |
| `independent_sum` | forced choices + sum of the children | sum of the children's bounds |
| `separator` | forced choices + best branch (separator assignment + its children) | the largest bound over **all** branches |

A component is `optimal` only when every leaf it uses is exact and no other branch could do better (its best branch reaches the bound). The final schedule is checked for feasibility: at most one slot per satellite and no conflicting pair.

In [ ]:
phase_started = time.perf_counter()

def solve_terminal_exact(leaf: dict[str, Any]) -> tuple[float, set[str]]:
    """
    Exact optimum of a terminal: branch and bound over its satellites (each takes one of its
    slots or none), pruning a branch when even the heaviest remaining slots cannot beat the best.
    """
    sub_csp = terminal_csp(leaf)
    sats = sorted(sub_csp.vars, key=lambda v: (len(v.domain), v.name))
    options = [sorted(v.domain, key=lambda t: (-SLOT_WEIGHT[t.slot_id], t.slot_id)) for v in sats]
    # optimistic value of the satellites from position i on (heaviest slot each, conflicts ignored)
    rest = [0.0] * (len(sats) + 1)
    for i in range(len(sats) - 1, -1, -1):
        rest[i] = rest[i + 1] + (SLOT_WEIGHT[options[i][0].slot_id] if options[i] else 0.0)
    best = {"value": 0.0, "slots": []}

    def dfs(i: int, chosen: list[Timeslot], blocked: frozenset, value: float) -> None:
        if value + rest[i] <= best["value"]:
            return
        if i == len(sats):
            best["value"], best["slots"] = value, list(chosen)
            return
        for t in options[i]:
            if t not in blocked:
                chosen.append(t)
                dfs(i + 1, chosen, blocked | sub_csp.index.conflicts[t], value + SLOT_WEIGHT[t.slot_id])
                chosen.pop()
        dfs(i + 1, chosen, blocked, value)      # the satellite stays unscheduled

    dfs(0, [], frozenset(), 0.0)
    return best["value"], {t.slot_id for t in best["slots"]}


def evaluate_node(node: dict[str, Any], memo: dict[str, dict[str, Any]]) -> dict[str, Any]:
    """Evaluate one tree node bottom-up (same combination rules as the MWIS evaluate_node)."""
    node_id = node["decomposition_node_id"]
    if node_id in memo:       # shared sub-problems (cache hits) are evaluated once
        return memo[node_id]
    base_selected = set(node["forced_selected"].values())
    base_objective = float(node["forced_objective"])

    if node["node_type"] == "incomplete":
        heuristic = node["heuristic_solution"]          # already includes the forced choices
        output = {
            "status": "heuristic_feasible",
            "selected_nodes": set(heuristic["selected"].values()),
            "objective": heuristic["objective"],
            "upper_bound": heuristic["upper_bound"],
            "terminal_ids": set(),
            "heuristic_ids": {node_id},
            "fallback_details": [{
                "node_id": node_id,
                "component_id": node["original_component_id"],
                "reason": node["reason"],
                "qubits": node["qubits"],
                "selected_nodes": json.dumps(sorted(heuristic["selected"].values())),
                "heuristic_objective": heuristic["objective"],
                "upper_bound": heuristic["upper_bound"],
            }],
        }

    elif node["node_type"] == "terminal":
        value, selected = solve_terminal_exact(node) if node["qubits"] > 0 else (0.0, set())
        output = {
            "status": "optimal",
            "selected_nodes": base_selected | selected,
            "objective": base_objective + value,
            "upper_bound": base_objective + value,
            "terminal_ids": {node_id} if node["qubits"] > 0 else set(),
            "heuristic_ids": set(),
            "fallback_details": [],
        }

    elif node["node_type"] == "independent_sum":
        children = [evaluate_node(child, memo) for child in node["children"]]
        output = {
            "status": "optimal" if all(c["status"] == "optimal" for c in children) else "heuristic_feasible",
            "selected_nodes": base_selected.union(*(c["selected_nodes"] for c in children)),
            "objective": base_objective + sum(c["objective"] for c in children),
            "upper_bound": base_objective + sum(c["upper_bound"] for c in children),
            "terminal_ids": set().union(*(c["terminal_ids"] for c in children)),
            "heuristic_ids": set().union(*(c["heuristic_ids"] for c in children)),
            "fallback_details": [d for c in children for d in c["fallback_details"]],
        }

    else:   # separator: every branch is evaluated, the best one is kept
        branch_results = []
        for branch in node["branches"]:
            children = [evaluate_node(child, memo) for child in branch["children"]]
            fixed = {slot_id for slot_id in branch["assignment"].values() if slot_id is not None}
            branch_results.append({
                "status": "optimal" if all(c["status"] == "optimal" for c in children) else "heuristic_feasible",
                "selected_nodes": base_selected.union(fixed, *(c["selected_nodes"] for c in children)),
                "objective": base_objective + branch["state_objective"] + sum(c["objective"] for c in children),
                "upper_bound": base_objective + branch["state_objective"] + sum(c["upper_bound"] for c in children),
                "terminal_ids": set().union(*(c["terminal_ids"] for c in children)),
                "heuristic_ids": set().union(*(c["heuristic_ids"] for c in children)),
                "fallback_details": [d for c in children for d in c["fallback_details"]],
            })
        output = dict(max(branch_results, key=lambda r: (r["objective"], sorted(r["selected_nodes"]))))
        # the bound must cover every branch, also the ones not selected
        output["upper_bound"] = max(r["upper_bound"] for r in branch_results)
        if output["upper_bound"] > output["objective"] + 1e-9:
            output["status"] = "heuristic_feasible"

    memo[node_id] = output
    return output


memo: dict[str, dict[str, Any]] = {}
component_results = [evaluate_node(d["root"], memo) for d in decompositions]

selected_nodes = set().union(*(r["selected_nodes"] for r in component_results))
final_objective = sum(r["objective"] for r in component_results)
final_upper_bound = sum(r["upper_bound"] for r in component_results)
solution = {
    "status": "optimal" if all(r["status"] == "optimal" for r in component_results) else "heuristic_feasible",
    "selected_nodes": selected_nodes,
    "objective": final_objective,
    "upper_bound": final_upper_bound,
    "absolute_gap_bound": final_upper_bound - final_objective,
    "relative_gap_bound_percent": 0.0 if final_upper_bound == 0 else 100 * (final_upper_bound - final_objective) / final_upper_bound,
    "exact_component_count": sum(r["status"] == "optimal" for r in component_results),
    "heuristic_component_count": sum(r["status"] != "optimal" for r in component_results),
    "exact_components_objective_sum": sum(r["objective"] for r in component_results if r["status"] == "optimal"),
    "heuristic_components_objective_sum": sum(r["objective"] for r in component_results if r["status"] != "optimal"),
    "heuristic_components_upper_bound": sum(r["upper_bound"] for r in component_results if r["status"] != "optimal"),
    "terminal_ids": set().union(*(r["terminal_ids"] for r in component_results)),
    "heuristic_ids": set().union(*(r["heuristic_ids"] for r in component_results)),
    "fallback_details": [d for r in component_results for d in r["fallback_details"]],
}

# sanity check: at most one slot per satellite and no conflicting pair in the final schedule
slot_by_id = {t.slot_id: t for v in csp.vars for t in v.domain}
chosen_slots = [slot_by_id[slot_id] for slot_id in solution["selected_nodes"]]
owners = [csp.index.owner[t] for t in chosen_slots]
if len(set(owners)) != len(owners):
    raise AssertionError("A satellite takes more than one slot in the evaluated schedule.")
if any(c.slot_id in solution["selected_nodes"] for t in chosen_slots for c in csp.index.conflicts[t]):
    raise AssertionError("The evaluated schedule contains conflicting slots.")

record_phase_timing("classical_evaluation", phase_started, objective=round(final_objective, 3), upper_bound=round(final_upper_bound, 3))
print({
    "status": solution["status"],
    "scheduled_satellites": len(solution["selected_nodes"]),
    "objective": round(solution["objective"], 3),
    "upper_bound": round(solution["upper_bound"], 3),
    "relative_gap_bound_percent": round(solution["relative_gap_bound_percent"], 3),
    "exact_components": solution["exact_component_count"],
    "heuristic_components": solution["heuristic_component_count"],
    "heuristic_leaves_used": len(solution["heuristic_ids"]),
})

## 9. Export

The artifacts are saved in `RUN_DIR`:

| File | Content | Read by |
|---|---|---|
| `csp_variables.csv`, `csp_constraints.csv` | the original CSP (domain size and degree per satellite, one row per constraint) | graphics notebook |
| `csp_components.csv` | the components after dominance, with their decomposition status | graphics notebook |
| `conflict_graph_nodes.csv`, `conflict_graph_edges.csv` | the reduced problem as a slot-level conflict graph (same files as the MWIS pipeline): one node per slot left after dominance, with its weight and `component_id`; one edge per pair of slots that cannot be selected together (antenna conflict or same satellite) | MILP fallback of the annealing notebook |
| `decomposition_summary.csv` | one row per component (section 5) | reports, annealing notebook |
| `csp_terminal_subproblems.json` | every non-empty terminal: its domains (`{satellite: [slot_id]}`), forced choices and size | reports |
| `csp_terminal_qubos.json` | every non-empty terminal: its QUBO (`[i, j, coefficient]` rows), `index_map` and forced choices (section 7) | annealing notebook |
| `csp_solution.csv` | the baseline schedule of section 8 (one row per selected slot) | reports, comparison with annealing |
| `component_csp_summary.csv` | one row per component: status, objective, upper bound, exact (same columns as `component_mwis_summary.csv`) | annealing notebook, MILP fallback |
| `heuristic_fallback_details.csv` | the incomplete leaves used by the baseline schedule | reports |
| `reconstruction_metadata.json.gz` | every tree node in the MWIS reconstruction schema (terminals point to `csp_terminal_qubos.json`) | annealing notebook |
| `csp_incomplete_leaves.json` | every incomplete leaf: its domains and its heuristic solution | reports |

The terminals are saved both as sub-problems (domains) and as QUBOs. The reconstruction metadata follow the MWIS schema (`producer: csp_pipeline`): node types, children, separator branches (`selected_separator` holds the slots fixed by the branch, `assignment` also the satellites left unscheduled), forced selections, terminal ids and heuristic solutions. Node sizes are in QUBO variables (`qpu_node_count`) and couplers (`qpu_edge_count`, known only for nodes sized for the QPU).

In [ ]:
phase_started = time.perf_counter()

# one row per satellite of the original CSP: domain size and degree
variables_rows = [
    {"var_id": v.id, "name": v.name, "domain_size": len(v.domain), "degree": csp.degree(v)}
    for v in csp.vars
]
pd.DataFrame(variables_rows).to_csv(RUN_DIR / "csp_variables.csv", index=False)

# one row per constraint (satellite pair with conflicting slots)
constraint_rows = [
    {"u_id": v.id, "v_id": u.id}
    for v in csp.vars
    for u in csp.neighbors(v)
    if u.id > v.id
]
pd.DataFrame(constraint_rows).to_csv(RUN_DIR / "csp_constraints.csv", index=False)

# one row per connected component after dominance (largest first)
components_rows = [
    {
        "component_id": decomposition["component_id"],
        "num_vars": len(component.vars),
        "num_constraints": component.num_constraints(),
        "qubo_vars": sum(len(v.domain) for v in component.vars),
        "decomposition_status": decomposition["status"],
    }
    for component, decomposition in zip(components, decompositions)
]
pd.DataFrame(components_rows).to_csv(RUN_DIR / "csp_components.csv", index=False)

# slot-level conflict graph of the reduced problem (same files as the MWIS pipeline, used by the
# MILP fallback of the annealing notebook): one node per slot left after dominance, one edge per
# pair of slots that cannot be selected together (antenna conflict, or two slots of one satellite)
slot_rows = {row["slot_id"]: row for row in slots.to_dict(orient="records")}
graph_node_rows, graph_edge_rows = [], []
for component, decomposition in zip(components, decompositions):
    for var in component.vars:
        for t in sorted(var.domain, key=lambda t: t.slot_id):
            row = slot_rows[t.slot_id]
            graph_node_rows.append({
                "slot_id": t.slot_id, "satellite_id": var.name, "antenna_id": t.antenna_id,
                "start_time_hours": row["start_time_hours"], "end_time_hours": row["end_time_hours"],
                "weight": SLOT_WEIGHT[t.slot_id], "component_id": decomposition["component_id"],
            })
live_slots = {t for var in reduced_csp.vars for t in var.domain}
for var in reduced_csp.vars:
    domain = sorted(var.domain, key=lambda t: t.slot_id)
    for t in domain:
        for u in reduced_csp.index.conflicts[t]:
            if u in live_slots and t.slot_id < u.slot_id:
                graph_edge_rows.append({"source": t.slot_id, "target": u.slot_id, "reason": "same-antenna conflict"})
    for t, u in combinations(domain, 2):
        graph_edge_rows.append({"source": t.slot_id, "target": u.slot_id, "reason": "same satellite"})
pd.DataFrame(graph_node_rows).to_csv(RUN_DIR / "conflict_graph_nodes.csv", index=False)
pd.DataFrame(graph_edge_rows).to_csv(RUN_DIR / "conflict_graph_edges.csv", index=False)

# leaves of the decomposition trees, in a CSP-native format
terminal_subproblems = [
    {
        "terminal_id": n["decomposition_node_id"],
        "original_component_id": n["original_component_id"],
        "depth": n["depth"],
        "qubits": n["qubits"],
        "couplers": n.get("couplers"),
        "terminal_reason": n["reason"],
        "forced_selected": n["forced_selected"],
        "forced_objective": n["forced_objective"],
        "domains": n["domains"],
    }
    for d in decompositions for n in d["terminals"] if n["qubits"] > 0
]
(RUN_DIR / "csp_terminal_subproblems.json").write_text(json.dumps(terminal_subproblems, indent=2), encoding="utf-8")

incomplete_export = [
    {
        "node_id": n["decomposition_node_id"],
        "original_component_id": n["original_component_id"],
        "depth": n["depth"],
        "reason": n["reason"],
        "qubits": n["qubits"],
        "domains": n["domains"],
        "heuristic_solution": n["heuristic_solution"],
    }
    for d in decompositions for n in d["incomplete"]
]
(RUN_DIR / "csp_incomplete_leaves.json").write_text(json.dumps(incomplete_export), encoding="utf-8")

record_phase_timing("export", phase_started, terminals=len(terminal_subproblems), incomplete=len(incomplete_export))
print(f"Exported CSP structure and sub-problems to: {RUN_DIR.resolve()}")

In [ ]:
phase_started = time.perf_counter()

# baseline schedule and per-component results (same columns as the MWIS component_mwis_summary.csv)
slots[slots["slot_id"].isin(solution["selected_nodes"])].to_csv(RUN_DIR / "csp_solution.csv", index=False)
component_csp_summary = pd.DataFrame([
    {
        "component_id": d["component_id"],
        "status": r["status"],
        "objective": float(r["objective"]),
        "upper_bound": float(r["upper_bound"]),
        "exact": r["status"] == "optimal",
    }
    for d, r in zip(decompositions, component_results)
])
component_csp_summary.to_csv(RUN_DIR / "component_csp_summary.csv", index=False)
pd.DataFrame(solution["fallback_details"]).to_csv(RUN_DIR / "heuristic_fallback_details.csv", index=False)


def reconstruction_node_metadata(node: dict[str, Any]) -> dict[str, Any]:
    """Compact record of a tree node, in the schema of the MWIS reconstruction metadata."""
    record = {
        "node_id": node["decomposition_node_id"],
        "component_id": node["original_component_id"],
        "node_type": node["node_type"],
        "depth": node["depth"],
        "input_node_count": node["input_qubits"],       # QUBO variables before the preprocessing
        "qpu_node_count": node["qubits"],               # QUBO variables after it
        "qpu_edge_count": node.get("couplers"),         # known only for nodes sized for the QPU
        "forced_selected": sorted(node["forced_selected"].values()),
        "forced_objective": float(node["forced_objective"]),
    }
    if node["node_type"] == "terminal":
        record["terminal_reason"] = node["reason"]
        record["terminal_id"] = node["decomposition_node_id"] if node["qubits"] else None
    elif node["node_type"] == "incomplete":
        heuristic = node["heuristic_solution"]
        record["reason"] = node["reason"]
        record["heuristic_solution"] = {
            "status": heuristic["status"],
            "forced_selected": sorted(heuristic["forced_selected"].values()),
            "heuristic_selected_nodes": sorted(heuristic["heuristic_selected"].values()),
            "selected_nodes": sorted(heuristic["selected"].values()),
            "objective": heuristic["objective"],
            "upper_bound": heuristic["upper_bound"],
        }
    elif node["node_type"] == "independent_sum":
        record["children"] = [child["decomposition_node_id"] for child in node["children"]]
    elif node["node_type"] == "separator":
        record["separator"] = list(node["separator"])           # satellites
        record["separator_method"] = node["separator_method"]
        record["branches"] = [
            {
                "state_index": branch["state_index"],
                # slots fixed by the branch (the MWIS selected separator nodes)
                "selected_separator": sorted(s for s in branch["assignment"].values() if s is not None),
                "assignment": branch["assignment"],               # satellite -> slot_id or None
                "state_objective": float(branch["state_objective"]),
                "children": [child["decomposition_node_id"] for child in branch["children"]],
            }
            for branch in node["branches"]
        ]
    return record


# reconstruction map: component roots plus every tree node (shared nodes are listed once)
seen_nodes: set[str] = set()
metadata_nodes = []
for d in decompositions:
    for node in d["nodes"]:
        if node["decomposition_node_id"] not in seen_nodes:
            seen_nodes.add(node["decomposition_node_id"])
            metadata_nodes.append(reconstruction_node_metadata(node))

reconstruction_metadata = {
    "schema_version": 1,
    "producer": "csp_pipeline",
    "run_name": RUN_NAME,
    "objective": "maximum-weight schedule, at most one slot per satellite and no antenna conflict",
    "qubo_encoding": QUBO_ENCODING,
    "terminal_qubos_file": "csp_terminal_qubos.json",
    "components": [
        {"component_id": d["component_id"], "root_node_id": d["root"]["decomposition_node_id"], "status": d["status"]}
        for d in decompositions
    ],
    "nodes": metadata_nodes,
}
with gzip.open(RUN_DIR / "reconstruction_metadata.json.gz", "wt", encoding="utf-8", compresslevel=6) as metadata_file:
    json.dump(reconstruction_metadata, metadata_file)

record_phase_timing("reconstruction_export", phase_started, nodes=len(metadata_nodes))
print(f"Reconstruction metadata: {len(metadata_nodes)} nodes, {len(reconstruction_metadata['components'])} components")

## 10. Final summary table

This table is the main compact output of the notebook.

In [ ]:
# Final metrics as (name, value, meaning)
final_summary_rows = [
    ("Candidate slots", len(slots), "Rows of the input dataset."),
    ("Satellites (CSP variables)", len(csp.vars), "One variable per satellite."),
    ("Constraints", int(csp_summary["num_const"]), "Satellite pairs with at least one conflicting slot pair."),
    ("Slots removed by dominance", dominated_slots, "Exact reduction (section 4)."),
    ("Slots after dominance", sum(len(v.domain) for v in reduced_csp.vars), "QUBO variables of the reduced problem."),
    ("Constraints after dominance", reduced_csp.num_constraints(), "Satellite pairs still with a conflicting slot pair."),
    ("Connected components", len(components), "Independent sub-problems after dominance."),
    ("Components decomposed exactly", sum(d["status"] == "complete" for d in decompositions), "No incomplete leaf in their tree."),
    ("Terminal sub-problems", len(terminal_subproblems), f"Non-empty leaves fitting the QPU limits ({QUBO_ENCODING})."),
    ("Largest terminal (qubits)", max((t["qubits"] for t in terminal_subproblems), default=0), f"QPU limit: {MAX_QPU_TERMINAL_NODES}."),
    ("Incomplete leaves", len(incomplete_export), "Leaves left to the heuristic fallback (section 6)."),
    ("Largest incomplete leaf (qubits)", max((n["qubits"] for n in incomplete_export), default=0), "Size the fallback has to handle."),
    ("Largest terminal QUBO (couplers)", max((t["couplers"] for t in terminal_qubos), default=0), f"QPU limit: {MAX_QPU_TERMINAL_COUPLERS}."),
    ("Exact components objective sum", solution["exact_components_objective_sum"], "Objective of the components certified optimal (section 8)."),
    ("Exact global objective", solution["objective"] if solution["status"] == "optimal" else "not certified", "Available only when the global optimum is certified."),
    ("Incomplete/heuristic components feasible sum", solution["heuristic_components_objective_sum"], "Feasible objective of the components not certified optimal."),
    ("Incomplete/heuristic components conservative upper bound", solution["heuristic_components_upper_bound"], "Upper bound of the components not certified optimal."),
    ("Total feasible objective / lower bound", solution["objective"], "Objective of the baseline schedule."),
    ("Total conservative upper bound", solution["upper_bound"], "Upper bound on the true optimum."),
    ("Absolute optimality-gap bound", solution["absolute_gap_bound"], "Upper bound minus feasible objective."),
    ("Relative optimality-gap bound (%)", solution["relative_gap_bound_percent"], "Gap as a percentage of the upper bound."),
    ("Separator states", sum(d["separator_state_count"] for d in decompositions), f"Branches created (budget {MAX_SEPARATOR_STATES} per component)."),
]

final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value", "meaning"])
phase_started = time.perf_counter()
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)
record_phase_timing("final_reporting_export", phase_started, metrics=len(final_summary))
record_phase_timing("pipeline_total_so_far", PIPELINE_STARTED, run_name=RUN_NAME)

print(f"CSP experiment saved: {RUN_NAME}")
final_summary